# SMDA · A clean research walkthrough
**Symbolic Mechanistic Data Attribution: Tracing Training Influence to Learned Behavioral Policies**  
Reza Habibi · Darian Lee · Magy Seif El-Nasr

This notebook is the entry point to the accompanying `SMDA-release.zip`. The reusable implementation lives in `src/smda/`.

**Default path:** download small pinned public artifacts → fit six Ridge policies on CPU → inspect one policy → reconstruct the 200/114/166 training subsets. No language-model download or API key is needed for this path.

**Status:** these are cached-artifact reanalyses with reconstructed splits, not certified reproductions of the paper's tables. Optional fresh GPU experiments appear at the end. See `docs/NOTEBOOK_AUDIT.md` for scientific discrepancies found during cleanup.

## 1 · Open the release
In Colab, upload `SMDA-release.zip` when this cell asks. For local Jupyter, open this notebook from inside the extracted repository. Nothing is uploaded to a public repository by this notebook.

In [ ]:
from pathlib import Path
import sys, os, subprocess, zipfile

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/smda/policy.py").is_file()), None)
if ROOT is None:
    from google.colab import files
    uploaded = files.upload()  # Choose SMDA-release.zip
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one SMDA-release.zip file.")
    destination = Path("/content/smda_release").resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            path = (destination / member.filename).resolve()
            if not path.is_relative_to(destination):
                raise ValueError("Unsafe archive member.")
        archive.extractall(destination)
    ROOT = destination / "SMDA-release"
    if not (ROOT / "src/smda/policy.py").is_file():
        raise ValueError("The uploaded ZIP is not the expected SMDA release.")
os.chdir(ROOT)
print("Repository:", ROOT.name)

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import json
import numpy as np
from IPython.display import display, Markdown
from smda import RidgePolicy, fit_policy, decompose_influence
from smda.data import read_rows, feature_matrix, pair_id
print("SMDA installed. NumPy", np.__version__)

## 2 · Check the math with a small example
The two pathways sum to the first-order coefficient change. This synthetic numerical check is a demonstration of the method, not a paper result.

In [ ]:
from smda.policy import ridge_weights
rng = np.random.default_rng(42)
X = rng.normal(size=(50, 5))
Y = rng.normal(size=50)
dX = rng.normal(size=X.shape)
dY = rng.normal(size=Y.shape)
result = decompose_influence(X, Y, dX, dY, alpha=5)
eps = 1e-5
numerical = (ridge_weights(X + eps*dX, Y + eps*dY, 5)
             - ridge_weights(X - eps*dX, Y - eps*dY, 5)) / (2*eps)
np.testing.assert_allclose(result.delta_w, numerical, rtol=1e-6, atol=1e-8)
print("Derivative check passed.")
print("Coefficient-alignment score:", round(result.score, 6))

## 3 · Download the public research artifacts
Downloads are pinned to immutable Hugging Face revisions and checked against SHA256 hashes. The evaluation cache has 7,826 rows; the full 11,000-prompt corpus is a different artifact. Source data contain safety-sensitive prompts and responses. This notebook displays coefficients and counts rather than prompt text.

In [ ]:
from download_artifacts import download
paths = download(ROOT / "configs/artifacts.lock.json", ROOT / "data/downloads")
print("Pinned public artifacts are ready.")

## 4 · Fit and save the symbolic policies
We use the paper's no-intercept Ridge specification (λ=5), an 80/20 stratified train/test split, and a threshold selected only on training rows. The full-set attribution policy is saved separately from the held-out classification model. All row memberships are recorded as prompt hashes.

The original `step0_new.json` was inaccessible without authentication. Therefore these deterministic split reconstructions are explicitly labeled as reanalysis. They are not the original paper split manifests.

In [ ]:
from fit_cached import run as fit_cached
RUN_DIR = ROOT / "runs/colab_cached"
metrics = fit_cached(paths["eval_cache"], RUN_DIR)
lines = ["| Experiment | n | Train accuracy | Test accuracy |", "|---|---:|---:|---:|"]
for row in metrics:
    lines.append(f"| {row['experiment']} | {row['n']} | {row['train']['accuracy']:.3f} | {row['test']['accuracy']:.3f} |")
display(Markdown("\n".join(lines)))

## 5 · Inspect a saved, reusable model
Each JSON model stores numeric feature IDs, weights, regularization, threshold, and provenance. Human-readable feature descriptions can change; model alignment always uses the numeric IDs.

In [ ]:
policy = RidgePolicy.load(RUN_DIR / "models/harmful_natural.json")
feature_labels = {int(r["feature_idx"]): r["label"] for r in read_rows(paths["feature_labels"])}
weights = np.asarray(policy.weights)
order = np.argsort(np.abs(weights))[::-1][:10]
lines = ["| Feature | Coefficient | Label |", "|---|---:|---|"]
for j in order:
    fid = policy.feature_ids[j]
    label = feature_labels.get(fid, "Unlabeled").replace("|", "/").replace("\n", " ")
    lines.append(f"| {fid} | {weights[j]:+.3f} | {label} |")
display(Markdown("\n".join(lines)))
print("Training-selected decision threshold:", round(policy.threshold, 4))
print("Saved model:", (RUN_DIR / "models/harmful_natural.json").relative_to(ROOT))

## 6 · Reconstruct the corrected training subsets
The old public quantitative subset has 113 pairs and an incorrect mask. The corrected criterion removes 47 harmful and 39 harmless pairs, retaining 114. The qualitative condition removes the primary rater's 34 flagged pairs, retaining 166.

In [ ]:
from prepare_subsets import run as prepare_subsets
CURATED_DIR = ROOT / "data/curated"
counts = prepare_subsets(paths["sft_full"], CURATED_DIR)
assert counts == {"full": 200, "quant_removed_corrected": 114, "qual_removed": 166}
print(counts)
expected_qual = {pair_id(r["prompt"], r["response"]) for r in read_rows(paths["sft_qual_reference"])}
actual_qual = {r["pair_id"] for r in read_rows(CURATED_DIR / "qual_removed.json")}
assert actual_qual == expected_qual
print("Qualitative membership matches the pinned public 166-pair subset.")

## 7 · Optional fresh influence experiment · GPU
Leave `RUN_GPU_INFLUENCE=False` for the CPU walkthrough. This optional stage loads gated Llama weights and needs a large-memory CUDA GPU, adequate CPU RAM, and accepted upstream access terms. Add an `HF_TOKEN` Colab Secret or environment variable.

The current public artifacts contain three harmless SFT/evaluation prompt overlaps. To make the fresh run disjoint, this cell explicitly removes those pairs, creating a **new 197-pair experiment**. It then runs only one pair as a smoke test. This is not a reproduction of the paper's 200-pair attribution run. Inspect the audit before scaling up.

The fresh code uses post-block-10 activations, contiguous prefix scoring, the paper's ReLU proxy and active gate, and exact weight restoration. These corrected semantics require new results.

In [ ]:
RUN_GPU_INFLUENCE = False
if RUN_GPU_INFLUENCE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[llm]"])
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except ImportError:
        pass
    from prepare_influence_data import run as prepare_disjoint
    from run_influence import run as run_influence
    disjoint = ROOT / "data/influence_disjoint.json"
    retained = prepare_disjoint(CURATED_DIR / "full.json", paths["eval_cache"], disjoint, drop=True)
    print("New disjoint training set:", retained)
    run_influence(ROOT / "configs/influence.json", paths["eval_cache"], disjoint,
                  ROOT / "runs/influence_smoke", limit=1)
else:
    print("GPU influence is disabled. CPU results are complete.")

## 8 · Optional full fine-tuning · separate expensive run
The paper uses full-parameter SFT with 3 epochs, batch 1, gradient accumulation 16, learning rate 2e-5, bf16 and AdamW 8-bit. The all-token SFT loss follows the original text-format training; influence uses response-only loss. No automatic model publication is included.

`qual_removed` is the paper's strongest fine-tuned condition on F1, but base has higher judged accuracy. Keep all comparisons. The following cell is disabled by default and trains only the selected subset when enabled. It does not resolve the historical overlap/encoder issues.

In [ ]:
RUN_SFT = False
SFT_SUBSET = "qual_removed"  # full, quant_removed_corrected, or qual_removed
if RUN_SFT:
    if SFT_SUBSET not in {"full", "quant_removed_corrected", "qual_removed"}:
        raise ValueError("Unknown subset.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[llm,train]"])
    from train_sft import run as train_sft
    train_sft(ROOT / "configs/sft.json", CURATED_DIR / f"{SFT_SUBSET}.json", ROOT / "runs" / f"sft_{SFT_SUBSET}")
else:
    print("Full SFT is disabled.")

## 9 · Evaluation and release handoff
Use `scripts/evaluate_model.py` for soft prefix metrics or 30-token generations and `scripts/score_judgments.py` for completed refusal/compliance judgments. Commands and the fixed judge subset are documented in `docs/REPRODUCIBILITY.md`. No paid judging API is invoked automatically.

The optional original SAELens recipe is `scripts/train_sae.py`. Retraining an SAE changes the feature basis; the historical feature IDs cannot simply be reused.

Your fitted CPU models, split hashes, and actual metrics are in `runs/colab_cached/`. The final release needs the author decisions in `docs/ACL_RELEASE_CHECKLIST.md`, including licensing and scientific reconciliation.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT / "runs/colab_cached_results"), "zip", RUN_DIR)
print("Saved CPU result bundle:", Path(archive).name)
# In Colab, download this ZIP from the Files panel if desired.